# Per-channel weight quantization for a linear layer

One scale for a whole weight matrix lets a single large row wreck the precision of every other row. **Per-channel** quantization gives each output channel (each row of an `nn.Linear` weight) its own scale, and the matmul still runs on integers because a per-row scale can be applied **after** the product.

Write `quantize_per_channel(W, x, bits=8)` in PyTorch.

## Inputs
- `W`: float64 `(out, in)`, a linear layer's weight (`y = x W^T`). `x`: float64 `(N, in)` activations, kept in float.
- `bits`: int in $[2, 8]$. Let $q_{\max} = 2^{\text{bits} - 1} - 1$ (127 for 8 bits).

## Per-channel quantization
For each row $c$: $s_c = \max_j |W_{cj}| / q_{\max}$ (use $s_c = 1$ for an all-zero row) and
$$Q_{cj} = \mathrm{clamp}\bigl(\mathrm{round}(W_{cj} / s_c),\, -q_{\max},\, q_{\max}\bigr),$$
rounding half to even (`torch.round`).

## Quantized matmul
Compute $y_q = (x\, Q^\top) \odot s$, i.e. multiply by the **integer** matrix first, then scale column $c$ of the result by $s_c$. (Do the product in float64; the point is where the scale goes.)

## Error report
Also build the **per-tensor** baseline: one scale $s = \max_{c,j} |W_{cj}| / q_{\max}$ (or 1 if `W` is all zero), the same rounding and clamping, and $y_t = (x\, Q_t^\top)\, s$. Report the max absolute error of each against the float result $y = x W^\top$.

## Output
A tuple `(Q, scales, y_q, err_channel, err_tensor)`: `Q` int8 `(out, in)`, `scales` float64 `(out,)`, `y_q` float64 `(N, out)`, and two Python floats.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def quantize_per_channel(W, x, bits=8):
    """Return (Q int8 (out,in), scales (out,), y_q (N,out), err_channel float, err_tensor float)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, -0.5], [0.02, 0.01]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[127, -64], [127, 64]], "dtype": "int8"}, {"__nd__": [0.007874015748031496, 0.00015748031496062991], "dtype": "float64"}, {"__nd__": [[0.49606299212598426, 0.030078740157480313]], "dtype": "float64"}, 0.003937007874015741, 0.003937007874015741]}}, {"a": [{"__nd__": [[2.0, -1.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[1.0, 2.0, 3.0], [0.5, 0.0, -1.0]], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[7, -4, 0], [0, 0, 0]], "dtype": "int8"}, {"__nd__": [0.2857142857142857, 1.0], "dtype": "float64"}, {"__nd__": [[-0.2857142857142857, 0.0], [1.0, 0.0]], "dtype": "float64"}, 0.2857142857142857, 0.2857142857142857]}}, {"a": [{"__nd__": [[-0.005751108303105836, -0.006220993166194774, 0.0008025479526353546, 0.012524723497116306, -0.0032389662147366176, -0.011016437567606532, -0.007987351061217753, 0.017774511121114445, -0.0035040986157726752, -0.011836250641849183], [-0.030228928789749844, 0.029827317660283587, 0.02871020714501013, 0.1863030510752766, -0.019139729793680397, -0.15456457340130703, 0.15088600403332872, 0.028700950327336022, 0.030000190493599067, -0.0679030494292641], [-1.0356338683299615, 1.5807695824286596, -1.5177162457649291, -0.6201256783781701, -1.146867109399292, 1.4436946534628072, 0.32228070741198306, -0.7082746784670266, 0.7131900347206027, 1.9445045519534288], [-4.813677481068201, -4.4676415689513185, 1.9998090090092482, 1.2131764235759692, -5.679434171998326, -3.1918380155317276, -0.09629714805261876, 4.3205695337241465, -7.435901151388327, 3.3603164559499428], [0.25388650427626364, 1.0294495503339425, -1.1986099625769966, -0.17071858270991688, -0.19718300387213752, 0.7688472565003824, -0.590347403997169, -0.27682298253387017, 0.13254662456401184, 0.3155531828262505], [0.025395377823587636, -0.06490868434698431, 0.09976777675555318, -0.034792001445607304, 0.023077193344650925, 0.049693373337620105, -0.01777884095117412, 0.028134138590302567, -0.07695678955626006, -0.006087592130231456]], "dtype": "float64"}, {"__nd__": [[-0.5469081575202287, -1.581464675415443, -0.7101176371329816, -1.0683301285402995, 0.9790498495005723, -0.8604312556681926, -1.6213092635863207, -0.22810207619529926, -0.37293733865626555, 2.686264288431484], [1.8651449017702242, 0.17307336674645923, -0.8335529684788846, 2.397183074310078, 0.2914595489529246, -1.8120982973548794, 0.4110612931132611, -1.8237792720158315, 1.3030167658735998, 0.4196593534117252], [-1.242308415293468, -0.6754737022321733, -2.3637309271316744, 1.8078080101470122, 0.061515944550117524, 0.6805455201036975, 1.365045970195392, -0.47239666846616, -0.6498909827907781, -0.5547359734554009], [0.2878461120037918, -0.9442685030865876, -0.422331907300688, 0.2600603396040004, 0.1126034765414661, -0.6562219631346289, -1.2211658256353948, -0.6543990822853525, 0.656762075166698, 1.4043867876130853], [1.2067264072707709, -0.3355963085080421, 0.670317155149419, 0.39270099148770127, 1.598841364174742, -0.8714250507302435, 1.2716514708970874, 0.4110044918535655, -0.16683684974119012, 2.6827703919064376]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-41, -44, 6, 89, -23, -79, -57, 127, -25, -85], [-21, 20, 20, 127, -13, -105, 103, 20, 20, -46], [-68, 103, -99, -41, -75, 94, 21, -46, 47, 127], [-82, -76, 34, 21, -97, -55, -2, 74, -127, 57], [27, 109, -127, -18, -21, 81, -63, -29, 14, 33], [32, -83, 127, -44, 29, 63, -23, 36, -98, -8]], "dtype": "int8"}, {"__nd__": [0.0001399567804812161, 0.001466953158073044, 0.015311059464200227, 0.05855040276683722, 0.009437873721078714, 0.0007855730453193164], "dtype": "float64"}, {"__nd__": [[-0.016436783377860625, -0.5794328066294898, 2.0542160775884346, 15.08597422418991, 0.2283661201232286, 0.06984023446727664], [-0.008758904399652805, 0.6618389416259659, -1.6826564250700553, -20.495329539445905, 0.35976192307997934, -0.375418274123676], [0.013686077914333364, 0.39066554422109445, 2.810342892027727, 4.7211600158471985, 1.0817079424545215, -0.2349206978358237], [-0.006953753024981702, -0.18079356362270824, 0.8860186006819957, 0.883568963104844, 0.46937798047209545, -0.0680614667938267], [-0.029085699403090116, 0.1678148896336707, -0.9223999175080931, 3.0685824223741363, -1.9460503453390872, 0.08344657062532541]], "dtype": "float64"}, 0.0845129042782844, 0.14837805291678863]}}, {"a": [{"__nd__": [[-0.005751108303105836, -0.006220993166194774, 0.0008025479526353546, 0.012524723497116306, -0.0032389662147366176, -0.011016437567606532, -0.007987351061217753, 0.017774511121114445, -0.0035040986157726752, -0.011836250641849183], [-0.030228928789749844, 0.029827317660283587, 0.02871020714501013, 0.1863030510752766, -0.019139729793680397, -0.15456457340130703, 0.15088600403332872, 0.028700950327336022, 0.030000190493599067, -0.0679030494292641], [-1.0356338683299615, 1.5807695824286596, -1.5177162457649291, -0.6201256783781701, -1.146867109399292, 1.4436946534628072, 0.32228070741198306, -0.7082746784670266, 0.7131900347206027, 1.9445045519534288], [-4.813677481068201, -4.4676415689513185, 1.9998090090092482, 1.2131764235759692, -5.679434171998326, -3.1918380155317276, -0.09629714805261876, 4.3205695337241465, -7.435901151388327, 3.3603164559499428], [0.25388650427626364, 1.0294495503339425, -1.1986099625769966, -0.17071858270991688, -0.19718300387213752, 0.7688472565003824, -0.590347403997169, -0.27682298253387017, 0.13254662456401184, 0.3155531828262505], [0.025395377823587636, -0.06490868434698431, 0.09976777675555318, -0.034792001445607304, 0.023077193344650925, 0.049693373337620105, -0.01777884095117412, 0.028134138590302567, -0.07695678955626006, -0.006087592130231456]], "dtype": "float64"}, {"__nd__": [[1.992737424069045, -1.686310926344033, 0.13997390705365448, 0.8070309042981835, 1.2173533761471307, -1.3102936203067854, 0.48416692664476635, 0.24961217938512528, -0.8406502612159998, 0.8671780759745665], [-0.5717668564153212, -0.04635826692719106, -1.0642025661374708, 0.5019604731550423, 0.5411787717584974, -0.3477159329376596, -0.9340402764675436, 0.0019263888768412184, 0.27609646641155305, 0.18788648849290165], [-0.27590766721749427, 1.1022161936576527, -0.37760011747333383, 0.06896936764678915, 1.7039995643017982, -0.927109086247135, 0.036712759048100424, 0.3134951474041088, -0.8302896093450514, -1.5125589808031141]], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[-2, -2, 0, 5, -1, -4, -3, 7, -1, -5], [-1, 1, 1, 7, -1, -6, 6, 1, 1, -3], [-4, 6, -5, -2, -4, 5, 1, -3, 3, 7], [-5, -4, 2, 1, -5, -3, 0, 4, -7, 3], [1, 6, -7, -1, -1, 4, -3, -2, 1, 2], [2, -5, 7, -2, 2, 3, -1, 2, -5, 0]], "dtype": "int8"}, {"__nd__": [0.0025392158744449206, 0.02661472158218237, 0.2777863645647755, 1.0622715930554754, 0.17122999465385666, 0.014252539536507597], "dtype": "float64"}, {"__nd__": [[0.010780665014109321, 0.22534630156973198, -7.92802341904759, 5.520820988266404, -2.984285984533506, 0.20673194078858173], [0.015732509810136165, -0.036457700160560796, 1.0073617869135834, -1.706256339122605, 1.3040253155666706, -0.1392253740564211], [0.028373398357808766, 0.2550521586467679, -4.435444990151722, -7.357183592775601, -0.18721450348708982, -0.04953667382437635]], "dtype": "float64"}, 1.1868033247490795, 1.1868033247490795]}}, {"a": [{"__nd__": [[-1.0677976760636527, -2.3420758754231135, -0.006815761123321509, -0.09086039480606323, 0.7608761287284633, 2.056154493713785, 0.18388885266158536, 0.21396051140314548, 0.5320733865122912, -0.5460323003748319, -1.6737757373813804, -0.842223865735649, 0.16125140056093593, -0.033247624252173376, 0.028213100754802753, -0.6845755435031904], [0.04171673287287739, -0.37289109536217213, -0.18530532905776734, -1.4223037137312675, -0.956455076878489, -0.008256783037621514, 1.63818616724587, 0.08179155706535961, 0.6347742153300381, 2.0460773832231505, 0.2866290952029204, -0.34788164185977866, -2.157292139477988, -0.3869590856864879, 1.3113657494669624, -1.1076994414741033], [-0.40452972324438446, -0.21416697730440187, 0.2250907065855229, -1.1632504879192325, 0.8542562708958751, 0.6166240164108543, -1.1302826095050147, -3.127065556227376, 0.6407890471495179, 0.13488219492691647, 0.01395675622294804, -2.0137690488085345, -0.9281610439329882, -0.3328430312388471, 0.2243771454829965, -0.14916932520794793], [1.0076525603831898, -1.274490955607433, -1.3715311568460162, -0.10365680264421748, 1.2813814459807102, -1.4707409100945195, -1.6125315762262045, 1.822115516115299, -0.8336336775427236, 0.08261288546868915, 1.4714751447214098, -1.1406374432192161, 0.07833101235698216, 0.5096641337057621, -0.20809896000716846, -0.3918152278812096], [-0.7730690768484798, -0.4830233990936503, 1.105681862478324, -2.4598122779979774, 0.40294628249770664, -0.05938143109839323, 0.021965839015122602, 0.015681955217573554, -0.07329521735808225, 0.6357713670048413, -2.2016065114447048, 0.7973386892123614, 0.7211654212991812, 0.04268756302743358, -0.11891114810361031, 0.9787241118740098], [2.1934240897249366, 1.7382320048126267, 1.4466317136162539, -1.9837299751742943, 1.8644628035768458, 0.10962424184676653, -0.6747642727307577, 0.3355833408616383, 1.3060854020818258, -1.8441484092820424, 0.8336984910415087, -0.5506142254086163, 0.12049888592471555, -0.31687401216657535, 0.8067504757082717, 0.09556821762412178], [-0.7549429644472085, -0.9325592601111846, -0.5712765114871794, 0.6240311994245095, -0.7991223461796885, 0.868608029253761, 1.332384316010685, 0.2901344913742402, -2.1413121903271293, 0.4487391302670965, 0.5897481682053621, 1.0043580153421434, 0.3219127147415236, 1.5367115896181183, 1.6234463997172484, 0.13040587329779205], [0.9018996721696027, 2.0874169122206405, 0.758455638480251, 0.5292584844323985, 2.1961179135197515, -2.0937346237112475, 1.5808693180279811, -0.27156405364695096, -2.253720279527513, 1.040934806120426, 1.5835213474142542, -0.009889334390746105, 0.6367341354482291, 1.2209435705953433, -0.4053606432859103, 0.3938374156107384]], "dtype": "float64"}, {"__nd__": [[0.8466071248278603, -0.6759977471934704, -0.04685464412884569, -0.9118230786969296, -1.0202223154325232, -0.09064143514542784, 0.45227308862877286, 0.25475516008088755, 0.6969286354783572, 0.8075539502583521, 0.7773864189244865, -0.5694561602051452, -0.3313841317324233, -1.3684386062730778, 1.6850379289563857, 0.952649905961553], [-0.3050693070164348, -0.0558433963716412, 0.9053038839234563, -0.9154144163120316, -0.46237699259831816, -0.8937602007415324, -0.02995732274115433, 1.3626040132045074, -0.5316472828267969, 1.3011578188937418, 0.37499052095685814, -0.27610626566346586, 0.8840526192596992, -1.1920039476928854, 0.16361216253623673, 0.13526682582100374], [2.3323961643665254, -0.42120325735038033, 0.28572099128417877, -1.2655238548995005, -1.0186754950356258, 0.7163978226457001, 1.8040700495814943, 1.3331219161880834, -1.545892931180536, -1.0635044856048976, -1.3488883140734471, -0.09011421666511614, 1.1555555535218727, 0.9346250146025387, 0.09768843618538005, 0.0267612877736765], [-2.463083829707561, -3.7556351568063664, -0.40561814812562236, -1.9417739879620506, -0.19895000506629712, -1.2582067297283859, 0.6186527479605518, -0.7773505905459352, -1.1422842668290634, -0.2892606327534454, 0.973746878079959, -0.5923742584791112, 1.2709818258379557, -1.456227396468799, -1.4796765731953982, -0.5222333070134684]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[0, -1, 0, 0, 0, 1, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0], [0, 0, 0, -1, 0, 0, 1, 0, 0, 1, 0, 0, -1, 0, 1, -1], [0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, -1, 0, 0, 0, 0], [1, -1, -1, 0, 1, -1, -1, 1, 0, 0, 1, -1, 0, 0, 0, 0], [0, 0, 0, -1, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0], [1, 1, 1, -1, 1, 0, 0, 0, 1, -1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, -1, 0, 0, 0, 0, 1, 1, 0], [0, 1, 0, 0, 1, -1, 1, 0, -1, 0, 1, 0, 0, 1, 0, 0]], "dtype": "int8"}, {"__nd__": [2.3420758754231135, 2.157292139477988, 3.127065556227376, 1.822115516115299, 2.4598122779979774, 2.1934240897249366, 2.1413121903271293, 2.253720279527513], "dtype": "float64"}, {"__nd__": [[-0.4497490806702415, 6.979751035949199, 0.9840906579988734, 3.260135069715606, 0.33068894632129237, -0.2089671138469418, 0.15405408421091463, -5.501976581449318], [-2.8407209857717537, 2.8711565683980242, -3.3975496832542182, 2.4061245592116727, 1.3293413331181498, -1.83233493354391, -1.127833038692093, 0.13571079815864978], [5.8235468243456765, 1.9878606018326546, -3.886966563303699, -1.816498149080473, 6.430963152986979, 4.302124569487391, 9.383821944261177, 1.7566016609091726], [3.5685777977355864, 0.0922048473856638, 4.283219396894966, 5.33432334275505, 2.381164970322866, -12.578271129430115, -2.5159710637044888, -3.1955994213720396]], "dtype": "float64"}, 4.559207772330972, 9.51591298539935]}}, {"a": [{"__nd__": [[0.5, -127.0, 63.5], [1.5, 2.5, -3.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0, -127, 64], [64, 106, -127]], "dtype": "int8"}, {"__nd__": [1.0, 0.023622047244094488], "dtype": "float64"}, {"__nd__": [[0.0, 1.5118110236220472], [-127.0, 2.5039370078740157], [64.0, -3.0]], "dtype": "float64"}, 0.5, 0.5]}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[-0.21156244906801633, 0.45531452102567793, -0.2793219532316209, -0.2426003473559475], [-1.701681518484609, 1.1335064739732048, -0.628763099705851, 0.10725092717674825]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0, 0, 0, 0], [0, 0, 0, 0]], "dtype": "int8"}, {"__nd__": [1.0, 1.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, 0.0, 0.0]}}, {"a": [{"__nd__": [[-0.0016087284684134513, -0.10928698326320485, -0.8326851409104072, -1.1759068868977987, 1.5826249106603973, 1.4096120355224917, 1.171920416261738, 0.22791812437947, -0.39107471784203834, -0.6452682907074804, 0.36810362182952266, -0.6558675180545163, -0.6796772657473956, 0.4039076414448529, -0.4408028393620358, 0.18855473413164073, -0.2051762631397362, -0.17392091488007153, 1.7881266559773483, -1.149519700368123, 1.4262318797808842, -0.03122719241350463, 0.2968092480930154, -0.06936206194395501, -0.20438160129073635, 0.19256506476460974, -1.086085874127845, 2.699202875450627, 0.4683193264350644, -0.2158699676842781, -0.19328078529911316, 0.3425495319668842], [-0.1292793067265168, 1.8278468922517963, 1.2496770631139213, 1.59438116060913, -1.3444244828728316, -1.5217394920429996, 0.8395288438783989, -0.5812805492002178, -0.6232475039434929, 0.3597099241531325, -0.4719328236812677, -0.41334682473956025, -0.15308244209982877, 0.634522451369437, -0.5263387785213094, 3.8940564904804655, 0.3089759663266721, -0.2525050313750808, -0.2796067060711181, 1.0145169658436604, 2.4432750748563157, 1.1515217929118786, -0.41116730619214614, -0.5657566735230909, -0.0029432279543601396, -0.5632402096456829, 0.03577347101958766, -0.16472980044323549, 0.5617563844712723, -0.00247185116643503, 1.6846886649922832, -1.0286124630815938], [0.4316310018367032, -0.46646745815656776, 0.10544991627352636, -0.4567041102696013, -4.06226645896461, 1.3331280245374597, 0.4456689680651822, -0.1283294938581599, 4.23199634478139, -4.1877152667610344, 0.04194122929551878, -1.5138836803543634, 0.02347700261984673, 2.5184948662712277, -1.7352109277443324, 0.0006096933064140322, 0.8943732827151694, 0.811665893326309, -0.601164567033185, -1.0231812535003824, 1.5079912349206488, -3.055810887490073, 0.0767926670368869, -0.3886130194109795, 0.7689804956132266, -2.6559052486283083, 0.2724640628232548, 0.17927777064390554, 1.315065191032723, -1.250326885645755, -3.0434442512561786, -0.6335249856053283], [-1.1219830578504006, -0.23716696355242356, -0.5065835859511464, 0.1939217025661458, 0.6307456491151436, -0.46200243441061084, 3.790586821895962, 6.496721656542721, -1.2505811359882113, -0.3577367193808363, -1.5096714718965694, 0.5751069154706427, -0.7919394821205508, -0.14101530369324772, 2.6822588333553807, 0.12547578110697807, 0.4351856153708337, -0.9818598140867626, 0.10810635902978033, -2.5539564278739575, 1.009899572999262, -1.0287095496054794, -1.6907569106264246, -0.6855813800196287, 1.4869380693726937, -0.010032980758180508, -0.29262101352439823, -1.446609511028142, 0.16492044442225984, -0.7654099646853675, 0.13275890015696065, 1.6623215338026416], [0.05409369904389546, -0.47929643438320535, 1.2267719830614936, -2.611380481832333, 1.0270644850043595, 1.5952302446253743, -0.3470757161132532, 0.6507099015414507, 1.020812166893004, -1.4001670941753963, -9.96773407050815, -0.610334095633856, 1.3076321233330124, 2.4341626520600537, -0.9285964514793781, -1.0529074537637526, -0.2187734165642599, -1.2077824083555666, 0.45329975104522224, 0.9481945880698489, -0.6401823669002101, -0.6799362609571148, 0.6858973173138265, -0.9307018703895601, 2.2178139606058034, 0.2230774240274335, -0.036935583206161506, 0.9595958697033254, -1.8690721199635498, 0.6978203911900632, -0.640628096403622, 1.3686196967572042], [-0.2305902981237302, -1.0264807017278639, 0.19288107737793772, -1.7658419311039621, 0.39001382063145434, 0.06887791881322664, 1.5500244570174375, 0.6463011906166521, -0.04041912682022723, 2.1618200805330674, -0.5836758507384363, -3.8361533927773337, 0.7737693362971382, 0.1593958970169731, 0.7741045558311229, 0.7653411310363967, -0.6026570684182844, -2.591905513358487, -0.07937191518019315, -1.2615305303432507, -0.6480774371236159, -1.341354947796356, 0.7449713704575937, -1.2814559800172203, 0.1434964024818461, -2.971431972086262, -1.1577890503391361, -2.2496882395873383, 2.0553218769031814, -1.3833113763921698, -0.5047788825996983, 0.312948765082014], [1.2742179600135435, -2.7647618428016156, 1.86952383408748, 0.4328652364354719, -0.613020150544235, -2.6589927045830075, 1.9889010212741947, 1.42219004199612, 0.2418459740589816, -1.2492851446367692, -0.9999046269121524, 0.9613926912465477, 0.45701445393499146, 0.6715661022870568, -0.15428979642181254, -1.3399443682026346, -1.972847784321021, -0.42083217294165837, -1.2004278770441588, 2.452501812142526, -0.11542524348803514, 5.039028128015393, -0.6026917442407148, -1.0755648293854378, -0.6019773302037172, 3.2528710742828775, -0.12299977735271059, -2.8379902554780347, -2.0489742169579888, -0.12928870750427646, 0.25059375994748345, 0.5009481589289319], [4.095381024751924, -0.6339906402320055, 0.3232965563877768, -0.4134077256187573, 0.3527229217221266, -0.27594558946446957, 1.120345910037488, 0.22337040959068286, -0.7614426787145461, 0.7280681119257767, 0.03689182805663552, -1.9319911944056245, 0.13562301978618965, 0.465537972819411, 3.755652867530781, -2.9934013419401393, -0.40266535340887816, -0.1644872730926053, 0.12661174231212027, -0.3306896910295603, 1.1270097770483989, -0.8251744721003293, 0.7833025455074519, 0.963678710222426, -1.4266463328625048, 2.064471972632693, -0.6286314163161378, -1.212512567977859, -0.2810370893867437, 0.5473560234133276, 0.4968271812783303, 0.4231262160324447], [1.1566051813041185, -0.17453259961954923, 0.20972788353149438, 1.205416431937033, -1.4851482852388476, 0.7361905073176218, 1.207721623732005, -1.1299040779700147, -1.0337678456867412, 0.9471500099190767, -1.7391817896556783, 2.0816839826472697, -0.09036174653459628, -0.7815136343308192, -1.1577817908701358, -0.5190972732038669, -0.8234705529055939, -5.299073049323929, -3.0069540089862645, 0.4037253272571143, 0.05395093521430259, -0.627277676735613, -0.4086144254656327, -0.5677141345330943, -0.4060078993501905, -2.2032865210910733, -0.5874984392649936, 1.2114830169657809, -1.1451357355178677, 0.313834358147129, -0.2129492523089178, -0.6259325117492525], [0.3749708338585135, 0.13729242687273344, 1.2601256009390207, 0.5262724223948035, -1.5288656921444261, 3.0962202387548077, 0.31255523937147195, 4.042703766630713, -0.025512348711098674, 0.0778322750540646, -1.6638894336512176, 0.8201560530489365, -0.1496416058806529, -0.09156663878356883, 0.6008214266776917, -0.7611038031265127, -0.6615217089075413, -3.576833183731319, -0.08617389414373192, -0.3434302453732676, 0.22775558854299957, -0.13483491433134578, 0.18576127633669395, -3.341648300826543, 0.5144386213623034, 0.25791210443819645, 1.0836678615843032, 0.2898066940631893, -0.05956826438092271, -0.2947390487399561, 0.7359346030750291, -0.9809943647648864], [-2.335092739595696, 2.800999068894195, -2.389259758946405, -0.2065746307369421, -1.1294943400925528, 0.11765393409054462, -0.13044719918636838, 0.31922151037401375, 1.3896716813796228, -0.7264987593311464, 1.586588159014252, -1.1155616459690403, 0.05341319152060959, -0.6762418386543294, -0.33373493467780135, 3.045832586643787, -0.821399840698828, -1.425727896617439, -0.6287571501609822, 1.0021682755498036, 0.5713578577238642, 0.07737888071231608, 0.5356290430944517, 0.18802969055312677, 4.469998389622669, 1.9676843887904114, -0.5619304970171755, -0.39299872068365777, -0.04450525551677505, -0.9147910029807426, -3.377227936078957, -0.7104704861760471], [2.2024125752019565, -0.36251251176715626, 0.97151172031078, 2.343277187298308, 1.964188401612091, 0.9441128707316974, -0.7731399959091058, 0.11139505271500864, -2.970716699541129, 0.49817850686515025, -0.26867102820062105, 1.1125596774221296, -1.5499122247657324, 0.5562742700266986, 0.5123151245972569, -0.9707211667232489, -0.9930824576510645, -0.8794963016104573, 3.9833777518649716, 1.7493709151716283, -0.5460237074653443, -2.2069932293011245, 0.6272566408362532, -0.5742055912972726, -0.5854751640533258, -0.7458388979766415, 0.955444600753037, -0.21540748402436455, -6.718545215578404, -0.022589489883290022, 0.32979566467929583, 0.10075175431231126], [0.5430233019541197, 1.4609119561115473, 0.2967723995342791, 2.0757617974458986, -0.4143998535823449, -3.072104085598509, 0.3422305382557177, -0.8574238263646062, -0.8585835215993405, 1.197545456671352, 1.1652523329094382, -1.4726459446894138, -0.7157670386286696, 2.207861737752132, -3.4384594300253375, 0.9571087534627127, -1.368928660920964, -2.1526864317025303, -2.5225527744953697, 0.7569208031679149, -0.4978482796801564, -0.8814894363728045, -0.15671454612813002, -0.10595253539056758, -1.1228609712439996, 0.3847684958997779, 0.8028200631740146, 1.0485820773223293, 1.494860297684453, -0.5819528502592067, 0.5262641704328898, -0.1916620350969352], [-0.3399711929011868, -0.4319449415123639, 0.438177165202518, 0.4890640813185179, -0.5903903953174545, 0.5148343839431097, -0.5033347751985253, 1.1817099199728192, -0.748308069776716, -1.3558614127330677, 1.2971667382687202, -0.8341558261013283, 2.5511953574080106, 2.54513387128926, 0.38009405174143773, 7.156446009459661, 0.057936847235827614, -0.10597217839389142, -0.8398149642142212, -0.48138804798048185, 0.1418464013745389, -0.7640622632972915, 1.5107310136843817, -0.631016207587582, -1.5584619281696255, 0.3970265638859661, -0.12386472868982494, 0.8804394389071011, -0.7771208228157565, -1.0466828506137997, 2.3731235871212304, -0.8963433953756885], [2.913815765406936, -0.3205460253056445, 0.04712569038334967, -1.0934983794389863, -0.8564406789342824, 1.5142994570346109, -2.026210548535789, 0.15432908124299988, 0.14513906888654832, -0.7701511231277752, -0.9120928122099048, 0.13097721062029116, -1.6825863053767212, 1.3458758420653627, 1.9014966368699697, -0.522597463147566, -0.00918828493473689, 0.8942696479807717, 0.9223715056864836, 1.8119579696210906, -1.9193148739935457, 0.8689339165886844, 0.6000554613654575, -1.548005800510732, 0.10365173008959895, -1.83229702033566, 3.822053461284725, -1.9622840723577204, -0.4862729362739247, -0.6004638445064248, 3.847569764993559, -0.2699182246730564], [0.17627315250347278, -3.1289517966648277, -1.0037696371622722, -0.4088127099108431, -0.8934306355910565, 1.607599492394382, -0.9230184921652486, 1.0017385543082984, -0.25697772470094477, -1.0020881635781715, 0.08700899447792833, -0.9773623670278236, -0.2955067176507977, -2.0171492511872136, -1.0131340537854183, -2.294332948043964, -1.0020269442212175, -1.7842893246879137, 0.3683110602240914, -1.0184519063391622, -1.1460105959176676, 2.4571429505570146, -0.48330182703062174, 0.419056175782261, -0.13412628809405053, -1.5551481036976513, 0.11451439355801425, -0.5621604731900637, -3.182959983314335, -1.5300679163136348, -1.6884737993710972, -1.0136024303053064]], "dtype": "float64"}, {"__nd__": [[1.1274786333048707, -0.39963288112541695, 0.2836603434996685, -0.06666117927661833, 0.10238644562421377, 1.7703442827394902, -1.1005956584015515, 1.098710413116607, -1.329257201922165, -0.563496261437578, -0.918010385582171, 1.4282755142491146, 0.25603418980913156, 0.6795765718318784, 0.25985102021435585, 0.4306673478825177, -1.260540291569246, 2.017785270050785, 1.161626709632629, 0.7333064684492222, 0.21638754643414596, 0.29920599716854634, -0.42013380012217344, -0.7941568167187197, 1.056121060203946, 1.0258877326036737, 0.9874639403928149, 0.5471513314905339, -2.0891199489001466, 1.4862248390057817, -0.8971136362471884, -0.1915064347214033], [-1.9985550868924549, 0.4919066199813112, -1.19118065742389, 0.08024225366424662, 1.758494262278282, -0.2434189670246784, 0.822363935385686, -0.16713000366304454, 1.830664535723283, -1.3600448117629036, -2.449787986953968, -0.042677279878042006, -0.12958079945367804, -0.6928036177104437, 0.9503617742607411, 2.1297514559463258, -0.2533959947259808, 1.1421689310626937, 0.30960469956394343, 0.022559116598301024, -0.2922084645460226, 0.04944057611474422, 0.3137150225112982, 0.3055294967226918, 2.0745943347843117, -0.02464768832484521, -1.1205343636571208, -0.805091572841477, 0.2837105858943159, 1.0070653504773863, -0.7871624882788023, 0.9342268997004667], [0.6464193195127955, 0.33077783279078266, 0.6746712470801847, -0.7705854347419735, -0.4838814983045842, 0.8965503181455388, -0.7717556531960968, 0.6604299320001892, -0.7972871387317438, 2.4013394144479485, -1.5901936786326913, 1.3398997020616665, 0.399577305571005, -0.15976358612853395, -0.5080654818686144, -1.4137988769957803, 0.46282438171472234, -0.7724978875642367, -1.362061634818601, 0.723556337240202, -0.6249869002258496, -1.176698659453066, -0.8479702848701913, 1.0408146216968734, -1.926957756842477, 0.3630738619566178, 0.807627770827224, 0.11313824372737796, -0.0625948587617214, -1.0274671662470407, -0.5963116646165664, -1.395163523503108], [-0.16988596688043145, -1.4531601969672638, 0.7405949005534666, 0.3821319295366239, -0.8702622957043932, 0.6362983433295245, -1.1292474571428055, -0.6821647981113403, -1.265972520598159, 1.003523423884619, -0.004154633738030377, -0.5066671087917571, -0.3221029370099168, -0.4093175709253875, -0.5616063612454939, 0.6484621307416418, 0.15171457166171848, -0.9516839461666641, 0.15465329616655005, 0.02111037688864483, -0.46707458674312136, 0.05101897484704513, 0.8928216223098013, 1.0817904876882458, -0.5601528119768785, 0.20901757394248105, -0.236298585955894, -0.328960504282723, 1.4111178817303969, 0.7735554481129284, 0.03804409348477955, -0.1617560994813792], [-0.9730114120721226, -0.7151433662487563, -0.32235617182059495, -0.6090558049393194, 0.9694580642552599, 1.5775212221734618, 2.502997078302826, -2.0752676312303797, -1.30240685856094, -1.4708721214007203, -0.45156735346692717, 0.21203523448628891, 0.479134042307869, -0.20089541291022042, -0.4611888291970646, -0.45063686808915954, -1.6635371339459957, 0.6773845302708443, -0.4252681130968972, -1.459617171687357, -1.0931352333878164, -0.8934990048134783, -0.8812034873372122, -1.8942068102250391, 0.022966735119038833, 1.672130960469506, 0.37365858343936703, -1.117040177660905, -0.5555490345730236, -1.043603636063559, 0.9000131182073516, 1.40707344669877], [0.8242731100318159, -0.6118100340104065, -0.46428495557664756, 1.0881764533021403, -0.2722538136581355, 0.9053989835956832, -2.054224107863984, -0.7893564286030973, 0.8549253238682112, 0.42820851044056635, 0.6430113601938925, 0.13928484202786004, -0.032591260279798716, -0.20410590660668326, 2.004095935236376, -2.604197173819406, 1.0485854498690703, -1.4411132749845676, -1.2029737890702812, 1.1764818925408285, 0.5264507028752301, -0.10606701673840128, -0.5118947964045443, 0.48055598958333756, 0.8426632156862451, -0.2937621284714452, -0.2653813290076588, -0.48201178158010977, 0.9317776462211832, -1.3049563283526555, -1.3591968368497707, 0.2605485798044435]], "dtype": "float64"}, 8], "e": {"__tup__": [{"__nd__": [[0, -5, -39, -55, 74, 66, 55, 11, -18, -30, 17, -31, -32, 19, -21, 9, -10, -8, 84, -54, 67, -1, 14, -3, -10, 9, -51, 127, 22, -10, -9, 16], [-4, 60, 41, 52, -44, -50, 27, -19, -20, 12, -15, -13, -5, 21, -17, 127, 10, -8, -9, 33, 80, 38, -13, -18, 0, -18, 1, -5, 18, 0, 55, -34], [13, -14, 3, -14, -122, 40, 13, -4, 127, -126, 1, -45, 1, 76, -52, 0, 27, 24, -18, -31, 45, -92, 2, -12, 23, -80, 8, 5, 39, -38, -91, -19], [-22, -5, -10, 4, 12, -9, 74, 127, -24, -7, -30, 11, -15, -3, 52, 2, 9, -19, 2, -50, 20, -20, -33, -13, 29, 0, -6, -28, 3, -15, 3, 32], [1, -6, 16, -33, 13, 20, -4, 8, 13, -18, -127, -8, 17, 31, -12, -13, -3, -15, 6, 12, -8, -9, 9, -12, 28, 3, 0, 12, -24, 9, -8, 17], [-8, -34, 6, -58, 13, 2, 51, 21, -1, 72, -19, -127, 26, 5, 26, 25, -20, -86, -3, -42, -21, -44, 25, -42, 5, -98, -38, -74, 68, -46, -17, 10], [32, -70, 47, 11, -15, -67, 50, 36, 6, -31, -25, 24, 12, 17, -4, -34, -50, -11, -30, 62, -3, 127, -15, -27, -15, 82, -3, -72, -52, -3, 6, 13], [127, -20, 10, -13, 11, -9, 35, 7, -24, 23, 1, -60, 4, 14, 116, -93, -12, -5, 4, -10, 35, -26, 24, 30, -44, 64, -19, -38, -9, 17, 15, 13], [28, -4, 5, 29, -36, 18, 29, -27, -25, 23, -42, 50, -2, -19, -28, -12, -20, -127, -72, 10, 1, -15, -10, -14, -10, -53, -14, 29, -27, 8, -5, -15], [12, 4, 40, 17, -48, 97, 10, 127, -1, 2, -52, 26, -5, -3, 19, -24, -21, -112, -3, -11, 7, -4, 6, -105, 16, 8, 34, 9, -2, -9, 23, -31], [-66, 80, -68, -6, -32, 3, -4, 9, 39, -21, 45, -32, 2, -19, -9, 87, -23, -41, -18, 28, 16, 2, 15, 5, 127, 56, -16, -11, -1, -26, -96, -20], [42, -7, 18, 44, 37, 18, -15, 2, -56, 9, -5, 21, -29, 11, 10, -18, -19, -17, 75, 33, -10, -42, 12, -11, -11, -14, 18, -4, -127, 0, 6, 2], [20, 54, 11, 77, -15, -113, 13, -32, -32, 44, 43, -54, -26, 82, -127, 35, -51, -80, -93, 28, -18, -33, -6, -4, -41, 14, 30, 39, 55, -21, 19, -7], [-6, -8, 8, 9, -10, 9, -9, 21, -13, -24, 23, -15, 45, 45, 7, 127, 1, -2, -15, -9, 3, -14, 27, -11, -28, 7, -2, 16, -14, -19, 42, -16], [96, -11, 2, -36, -28, 50, -67, 5, 5, -25, -30, 4, -56, 44, 63, -17, 0, 30, 30, 60, -63, 29, 20, -51, 3, -60, 126, -65, -16, -20, 127, -9], [7, -125, -40, -16, -36, 64, -37, 40, -10, -40, 3, -39, -12, -80, -40, -92, -40, -71, 15, -41, -46, 98, -19, 17, -5, -62, 5, -22, -127, -61, -67, -40]], "dtype": "int8"}, {"__nd__": [0.021253565948430135, 0.030661862129767445, 0.03332280586442039, 0.05115528863419465, 0.07848609504337127, 0.03020593222659318, 0.039677386834766876, 0.032247094683086015, 0.041724984640345894, 0.03183231312307648, 0.03519683771356432, 0.052901930831326015, 0.02707448370098691, 0.05634996857842253, 0.03029582492120913, 0.025062677033971144], "dtype": "float64"}, {"__nd__": [[1.73828422639753, -4.686919739939804, -6.702150270966505, 0.7969915072936635, 21.362343832977118, -22.39720715775944, 9.739248854170885, 0.5868271325452068, -7.506994725991804, 9.666196749528291, -1.7741820877535195, 28.811658580759538, -17.660974072101528, 2.715352322263469, 13.156841818194659, 4.676257988665678], [2.3150725427425645, 2.0364710530950063, 3.585113324023152, 13.81359150363363, 26.942296924218336, 2.6029256582603386, -3.982961447964766, -13.645612418758727, -13.485529772343911, -10.287762883343769, 22.726946066173657, -13.811568753260696, -16.454943714031167, 1.0122071162647526, -10.959173373150586, -10.531838214084138], [-7.683227421398424, -6.911323643807394, -7.970883817479708, -3.7079357926821115, 11.524703351304067, -0.5168927513688258, -2.4255183347812723, 5.493796736351735, 18.69240751515424, 11.254210441575275, -14.835416902246806, 6.108231464419446, 5.444175496269067, -10.145771327222812, 2.7982201275908194, 4.513684404581523], [-2.3842666702619413, 1.6609068067304231, -6.629698184051869, -12.196196805435212, -6.056461212856989, 6.351627823371488, -4.7412010519227135, -0.1547689316613609, 5.442864919421345, -0.9684238646088157, -5.088918014329824, -3.5554856678067934, 5.56443532916481, 3.910030631764959, -1.4328197958541047, 2.186364080518314], [4.717796375223832, -9.711889214271334, -4.616745498952964, 7.992943980323027, 10.483419717659288, 3.070918532703736, 8.922681903782715, 1.0838173753143219, -0.2751170594999433, 0.8814969723384577, -5.222115283035413, 3.8597610893324235, -4.395148803849017, -0.5224990223633802, -0.31100743988240326, 2.1951243175594644], [-7.572383829534796, -12.884408896726965, 7.542775304245577, -7.462761504847527, -6.981649287923695, 1.6941873837881893, -2.8167517980114214, 12.386991144556315, 8.980068719631104, 3.436578367031123, 3.2788966507500543, -3.0603488885017005, -5.41766707052009, -22.77834616995059, 4.39587728702676, 9.65199209531994]], "dtype": "float64"}, 0.33646235420506976, 0.37857891071527483]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(quantize_per_channel, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")